# Extract Database Structure from `views.csv`

This notebook uses `ViewsStructureExtractor` to:
1. read SQL view definitions from `data/views.csv`
2. extract structured metadata with an LLM
3. enrich each view with `source_tables_structure`
4. save the final payload to a JSON file in `data/`

In [1]:
import json
import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "Notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


from Classes.views_structure_classes import ViewsStructureExtractor

/Users/nikolajabramov/PycharmProjects/llm4lineage/.venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:27: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [2]:
# Configuration
csv_path = ROOT / "data" / "views.csv"
output_json_path = ROOT / "data" / "views_structure_result.json"

# Optional controls
limit = 10  # set to None to process all views
include_tables = None  # e.g. ["d_agr_collat_dmcl_attr", "d_agr_cred_dmcl_attr"]

print("CSV path:", csv_path)
print("Output path:", output_json_path)
print("Limit:", limit)
print("Include tables:", include_tables)

CSV path: /Users/nikolajabramov/PycharmProjects/llm4lineage/data/views.csv
Output path: /Users/nikolajabramov/PycharmProjects/llm4lineage/data/views_structure_result.json
Limit: 10
Include tables: None


In [3]:
hf_token = os.environ.get("HF_TOKEN")

if not hf_token:
    raise ValueError("HF_TOKEN is not set. Please export HF_TOKEN before running extraction.")

extractor = ViewsStructureExtractor(
    hf_token=hf_token,
    model="Qwen/Qwen3-Coder-30B-A3B-Instruct",
    provider="scaleway",
    max_new_tokens=2048,
    temperature=0.0,
    max_retries=3,
    llm_pause_seconds=0.0,
)

# Per-cycle observability callback
def _log_cycle(cycle: dict):
    print(
        f"[{cycle.get('index'):03d}] {cycle.get('view_name')} | "
        f"status={cycle.get('status')} | "
        f"elapsed={cycle.get('elapsed_ms')}ms | "
        f"sources={cycle.get('source_tables_count')} | "
        f"outputs={cycle.get('output_columns_count')}"
    )


result = extractor.extract_from_csv(
    csv_path=str(csv_path),
    limit=limit,
    include_tables=include_tables,
    progress_callback=_log_cycle,
    include_run_stats=True,
)

print("\nViews extracted:", result.get("views_count"))

[001] d_agr_collat_dmcl_attr | status=ok | elapsed=41949ms | sources=7 | outputs=8
[002] d_agr_cred_dmcl_attr | status=ok | elapsed=26993ms | sources=7 | outputs=9
[003] d_agr_cred_ifrs9 | status=warning | elapsed=367316ms | sources=10 | outputs=0
[004] d_agr_cred_qlty | status=ok | elapsed=39719ms | sources=5 | outputs=7
[005] d_agr_cred_prvsn | status=ok | elapsed=67569ms | sources=4 | outputs=5
[006] d_agr_cred_fin_f303_inf | status=ok | elapsed=19985ms | sources=4 | outputs=5
[007] d_agr_collat_mkt | status=ok | elapsed=50269ms | sources=6 | outputs=7
[008] d_agr_cred_agr_collat_core | status=warning | elapsed=6867ms | sources=1 | outputs=0
[009] d_agr_cred_fin_prt_coltp | status=error | elapsed=2639ms | sources=0 | outputs=0
[010] d_agr_cred_ovr_dp | status=warning | elapsed=6714ms | sources=1 | outputs=0

Views extracted: 10


## Validate source-tables structure and cycle logs

`ViewsStructureExtractor` now returns `source_tables_structure` directly from the `Classes` layer and exposes per-cycle run stats.

This step verifies both:
- structured source-table output
- cycle-level observability metadata (`run_stats`)

In [4]:
views = result.get("views", [])
run_stats = result.get("run_stats", [])

if not views:
    print("No views extracted.")
else:
    with_structure = [v for v in views if "source_tables_structure" in v]
    print(f"Views with source_tables_structure: {len(with_structure)}/{len(views)}")

    if run_stats:
        ok = len([x for x in run_stats if x.get("status") == "ok"])
        warn = len([x for x in run_stats if x.get("status") == "warning"])
        err = len([x for x in run_stats if x.get("status") == "error"])
        total_ms = sum(int(x.get("elapsed_ms", 0)) for x in run_stats)
        avg_ms = round(total_ms / len(run_stats), 2)
        print("\nCycle summary:")
        print(f"  total={len(run_stats)} | ok={ok} | warning={warn} | error={err} | avg_elapsed_ms={avg_ms}")

    first = views[0]
    print("\nExample keys:", list(first.keys()))
    print("\nFirst view name:", first.get("view_name"))
    print("First source tables count:", len(first.get("source_tables", [])))
    print("First source_tables_structure count:", len(first.get("source_tables_structure", [])))

    if first.get("source_tables_structure"):
        print("\nFirst source table structure:")
        print(json.dumps(first["source_tables_structure"][0], indent=2, ensure_ascii=False))

    if run_stats:
        print("\nFirst cycle log entry:")
        print(json.dumps(run_stats[0], indent=2, ensure_ascii=False))

Views with source_tables_structure: 10/10

Example keys: ['view_name', 'source_tables', 'source_tables_structure', 'output_columns', 'joins', 'filters', 'ctes']

First view name: d_agr_collat_dmcl_attr
First source tables count: 7
First source_tables_structure count: 7

First source table structure:
{
  "full_name": "s_grnplm_vd_t_bvd_db_dmcl.a_agr_collat_mkt_period",
  "schema": "s_grnplm_vd_t_bvd_db_dmcl",
  "table": "a_agr_collat_mkt_period",
  "columns_used": [
    "s_grnplm_vd_t_bvd_db_dmcl.a_agr_collat_mkt_period.host_eks_id",
    "s_grnplm_vd_t_bvd_db_dmcl.a_agr_collat_mkt_period.start_dt",
    "s_grnplm_vd_t_bvd_db_dmcl.a_agr_collat_mkt_period.end_dt",
    "s_grnplm_vd_t_bvd_db_dmcl.a_agr_collat_mkt_period.mkt_price_amt",
    "s_grnplm_vd_t_bvd_db_dmcl.a_agr_collat_mkt_period.mkt_price_rub",
    "s_grnplm_vd_t_bvd_db_dmcl.a_agr_collat_mkt_period.mkt_crncy_id"
  ],
  "join_conditions": [],
  "filter_references": []
}


In [5]:
# Save JSON output
output_json_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_json_path, "w", encoding="utf-8") as f:
    json.dump(result, f, indent=2, ensure_ascii=False)

print(f"Saved to: {output_json_path}")

Saved to: /Users/nikolajabramov/PycharmProjects/llm4lineage/data/views_structure_result.json


In [6]:
# Preview first extracted view
if result.get("views"):
    print(json.dumps(result["views"][0], indent=2, ensure_ascii=False)[:6000])
else:
    print("No views extracted.")

{
  "view_name": "d_agr_collat_dmcl_attr",
  "source_tables": [
    "s_grnplm_vd_t_bvd_db_dmcl.a_agr_collat_mkt_period",
    "s_grnplm_as_t_didsd_701_vd_dwh.v_crncy",
    "s_grnplm_vd_t_bvd_db_dmcl.d_prvsn_crncy",
    "s_grnplm_as_t_didsd_010_vd_dwh.v_crncy",
    "s_grnplm_vd_t_bvd_db_dmcl.a_agr_collat_qlty_period",
    "s_grnplm_vd_t_bvd_db_dmcl.d_agr_collat",
    "s_grnplm_vd_t_bvd_db_dmslcl.d_agr_collat"
  ],
  "source_tables_structure": [
    {
      "full_name": "s_grnplm_vd_t_bvd_db_dmcl.a_agr_collat_mkt_period",
      "schema": "s_grnplm_vd_t_bvd_db_dmcl",
      "table": "a_agr_collat_mkt_period",
      "columns_used": [
        "s_grnplm_vd_t_bvd_db_dmcl.a_agr_collat_mkt_period.host_eks_id",
        "s_grnplm_vd_t_bvd_db_dmcl.a_agr_collat_mkt_period.start_dt",
        "s_grnplm_vd_t_bvd_db_dmcl.a_agr_collat_mkt_period.end_dt",
        "s_grnplm_vd_t_bvd_db_dmcl.a_agr_collat_mkt_period.mkt_price_amt",
        "s_grnplm_vd_t_bvd_db_dmcl.a_agr_collat_mkt_period.mkt_price_rub",
   

## Notes

- Start with a small `limit` (e.g., `5` or `10`) to validate quality and runtime.
- Set `limit = None` to process all rows.
- Use `include_tables` to target a specific subset for debugging or iterative refinement.
- `source_tables_structure` is generated in `Classes/views_structure_classes.py` (not in notebook post-processing).
- Cycle observability is enabled via `progress_callback` and `include_run_stats=True`.
- Output file is written to `data/views_structure_result.json` by default.